# Inflation acceleration and deceleration

This notebook asks how inflation-sensitive assets behave after six-month changes in
real-time inflation momentum. It distinguishes the inflation rate from its direction
of change: a high but falling rate is economically different from a low but rising
rate.

Thresholds, series, horizons, and baselines were predeclared. Live provider values and
rendered results remain temporary; the committed narrative explains concepts and
procedures without asserting an observed result.

**Primary protocol.** The unit is the last common ETF trading session of a complete
month, with a one-calendar-day information lag. Acceleration is the treatment,
deceleration the reference, and one month the primary horizon. Outcomes are the paired
return spreads `TIP-IEF`, `DBC-IEF`, and `GLD-IEF`. Their contrasts are differences in
acceleration-minus-deceleration effects relative to nominal intermediate Treasuries,
so they directly operationalize “differ more.” Bonferroni intervals cover these three
primary outcomes. The stable state is descriptive; longer horizons, headline/core
sweeps, and revised history are exploratory.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from studies._support import (
    CORE_SYMBOLS,
    STUDY_START,
    acquire_latest_series,
    acquire_monthly_prices,
    acquire_vintage_histories,
    assert_component_periods_match,
    build_point_in_time_levels,
    classification_transition_table,
    compare_statistics,
    configure_plots,
    feature_provenance_summary,
    forward_labels,
    latest_revised_inflation_momentum,
    latest_revised_year_over_year,
    momentum_baseline,
    open_live_session,
    plot_coverage,
    plot_feature_comparison,
    plot_normalized_prices,
    plot_regime_contrasts,
    plot_regime_distributions,
    plot_regime_means,
    plot_regime_timeline,
    plot_revision_gap,
    plot_sample_sizes,
    plot_sensitivity_heatmap,
    point_in_time_inflation_momentum,
    point_in_time_year_over_year,
    regime_contrast_statistics,
    regime_statistics,
    regime_style,
    return_spread_labels,
    simultaneous_interval_family,
    study_run_manifest,
    temporal_contrast_stability,
    unconditional_statistics,
    validate_study_outputs,
)

configure_plots()
pd.set_option("display.max_columns", 20)
session = open_live_session()

## Hypothesis, inflation measures, and assets

`CPIAUCSL` is the primary headline consumer-price index. `CPILFESL`, which excludes
food and energy, is a predeclared measure sensitivity rather than a replacement
chosen after seeing results. Both series are revised and require vintage selection.

`TIP` extends the core asset set because inflation-linked Treasury principal responds
to measured inflation, while its market price also reflects real yields and duration.
The hypothesis is expressed through within-date spreads rather than comparing two
independent tables. A `TIP-IEF` outcome, for example, subtracts the two forward returns
before estimating the acceleration contrast and therefore preserves their covariance.
This is not a claim of inflation hedging at every horizon. TreasuryDirect's
[TIPS overview](https://www.treasurydirect.gov/marketable-securities/tips/) explains
inflation-adjusted principal, while market-price returns also reflect real yields,
duration, liquidity, and ETF mechanics.

In [ ]:
symbols = (*CORE_SYMBOLS, "TIP")
price_history, market_provenance = acquire_monthly_prices(session, symbols)
prices = price_history.loc[STUDY_START:]
series_ids = ("CPIAUCSL", "CPILFESL")
histories = acquire_vintage_histories(session, series_ids, prices.index)
latest = acquire_latest_series(session, series_ids)
staleness = {series_id: pd.Timedelta(days=62) for series_id in series_ids}
point_in_time = build_point_in_time_levels(
    histories,
    prices.index,
    staleness,
)
feature_provenance = feature_provenance_summary(point_in_time)
manifest = study_run_manifest(
    prices,
    series_ids=series_ids,
    thresholds="headline momentum band=±0.25 percentage points",
    staleness=staleness,
)
display(manifest, market_provenance, feature_provenance)

## Coverage before transformation

Inflation momentum needs four historical source components even though it produces one
value per decision. A matched current level does not guarantee that all exact comparison
months exist inside the same vintage. The coverage figure describes current-level
matching; component provenance and later regime counts reveal transformation coverage.
Because the transform opens the historical snapshot at each decision, it does not need
eighteen earlier decision rows or mix values captured on different dates.

No interpolation fills missing releases, and a deleted latest observation remains
missing under Persistra's point-in-time contract.

Observation period, source availability interval, and retrieval time have separate
meanings. The lagged decision cutoff must fall inside the source version's availability
interval, and the current observation must be no more than 62 days old. Market and CPI
objects round-trip through a temporary DuckDB database before analysis.

In [ ]:
figure, _ = plot_coverage(market_provenance, feature_provenance)
plt.show()
plt.close(figure)

## Point-in-time inflation rates

For each decision \(d\), year-over-year inflation is calculated within one as-of
vintage as \(100(C_{d,t}/C_{d,t-12}-1)\). The numerator and denominator are exact source
months from the same historical snapshot, not two entries on a decision-date sequence.
The latest-revised comparison uses today's values for those identical source periods.
This keeps the release calendar and staleness policy fixed while exposing revision
substitution.

Every component carries source identity, availability, frequency, unit, seasonal
adjustment, and retrieval provenance. The final audit checks that the point-in-time and
revised calculations requested the same economic months.

Headline and core inflation can diverge because volatile food and energy prices enter
only the headline measure. Neither series is treated as a direct forecast of asset
returns.

In [ ]:
point_rate_result = point_in_time_year_over_year(histories, point_in_time)
latest_rate_result = latest_revised_year_over_year(point_in_time, latest)
point_inflation = point_rate_result.frame
latest_inflation = latest_rate_result.frame
point_rates = point_inflation.rename(
    columns={"CPIAUCSL": "Headline inflation", "CPILFESL": "Core inflation"}
)
latest_rates = latest_inflation.set_axis(point_rates.columns, axis="columns")
figure, _ = plot_feature_comparison(
    point_rates,
    latest_rates,
    tuple(point_rates.columns),
)
plt.show()
plt.close(figure)

## Momentum and regime definition

Inflation momentum is the current year-over-year rate minus the year-over-year rate
for the source month six months earlier. All four levels—\(C_t\), \(C_{t-12}\),
\(C_{t-6}\), and \(C_{t-18}\)—come from one vintage known at the decision cutoff. Thus
\(M_d=100(C_t/C_{t-12}-1)-100(C_{t-6}/C_{t-18}-1)\). A value above 0.25 percentage
points is acceleration, below -0.25 is
deceleration, and the middle band is stable. The band avoids classifying negligible
changes as distinct macro states.

The primary regime uses headline inflation. Core inflation is reserved for sensitivity.
The timeline shows how smooth macro states can cluster; it also reveals that a
count of months can exaggerate the number of independent episodes.

The phase portrait makes level and momentum separate axes. A high inflation rate can
be decelerating, and a lower rate can be accelerating. Neither the six-month difference
nor the stable band imposes a separate persistence requirement.

In [ ]:
def momentum_regime(momentum: pd.Series, *, band: float = 0.25) -> pd.Series:
    regime = pd.Series(pd.NA, index=momentum.index, dtype="string")
    regime.loc[momentum.notna() & momentum.gt(band)] = "accelerating"
    regime.loc[momentum.notna() & momentum.lt(-band)] = "decelerating"
    regime.loc[momentum.notna() & momentum.between(-band, band, inclusive="both")] = (
        "stable"
    )
    return regime

point_momentum_result = point_in_time_inflation_momentum(
    histories, point_in_time
)
latest_momentum_result = latest_revised_inflation_momentum(
    point_in_time, latest
)
point_momentum = point_momentum_result.frame
latest_momentum = latest_momentum_result.frame
point_regimes = momentum_regime(point_momentum["CPIAUCSL"])
latest_regimes = momentum_regime(latest_momentum["CPIAUCSL"])
display(point_regimes.value_counts(dropna=False).rename("decision count"))
figure, _ = plot_regime_timeline(
    point_momentum["CPIAUCSL"],
    point_regimes,
    title="Real-time headline inflation momentum",
    ylabel="Six-month change in year-over-year inflation",
    boundaries=(-0.25, 0.25),
)
plt.show()
plt.close(figure)

figure, axis = plt.subplots(figsize=(9, 6))
for regime in ("accelerating", "stable", "decelerating"):
    color, marker = regime_style(regime)
    selected = point_regimes.eq(regime).fillna(False)
    axis.scatter(
        point_inflation.loc[selected, "CPIAUCSL"],
        point_momentum.loc[selected, "CPIAUCSL"],
        label=regime,
        color=color,
        marker=marker,
        alpha=0.75,
    )
axis.axhline(0, color="#333333", linewidth=0.9)
axis.set(
    title="Inflation level and momentum occupy different states",
    xlabel="Headline year-over-year inflation (percent)",
    ylabel="Six-month inflation-rate change (percentage points)",
)
axis.legend()
figure.tight_layout()
plt.show()
plt.close(figure)

## Outcomes and conventional baselines

Forward returns over one, three, and twelve decision months are calculated only after
the macro panel is complete. Each label retains its horizon end date and remains in a
different typed object. The final incomplete horizons stay missing.

The unconditional baseline represents ordinary behavior over the same dates. The
trailing twelve-month momentum split is a conventional time-series comparator for
each asset. Normalized prices show scale and crisis context but do not simulate
switching between inflation states.

For price \(P_d\), \(R_{d,h}=P_{d+h}/P_d-1\), and the stored ending close must fall
exactly \(h\) calendar months after the decision. The unconditional table is restricted
to valid headline states. One year of pre-analysis prices supplies trailing momentum
on the first study date. The relative-return labels subtract assets only after their
calendars and horizons are aligned.

In [ ]:
labels = forward_labels(prices)
eligible = point_regimes.notna()
unconditional = unconditional_statistics(labels, eligible=eligible)
momentum = momentum_baseline(price_history, labels, eligible=eligible)
display(unconditional, momentum)
figure, _ = plot_normalized_prices(prices)
plt.show()
plt.close(figure)

## Regime summaries and uncertainty

Counts and coverage accompany all conditional moments. A horizon-aware
heteroskedasticity-and-autocorrelation-consistent (HAC) interval accounts for the
mechanical overlap of multi-month outcomes, while the one-month
Persistra summary supplies annualized volatility and episode-aware drawdown. The
interval is approximate and does not make clustered inflation episodes independent.

The HAC bandwidth is the larger of \(h-1\) and a predeclared automatic rule. Group-mean
intervals are pointwise; Bonferroni intervals cover only the three primary one-month
relative-return contrasts. A gray point falls below twelve outcomes or two
outcome-eligible episodes on a side. That display rule does not make two episodes
sufficient for confident normal inference, so first/second-half and
leave-one-episode-out estimates remain central robustness evidence.

The complete table is the result family. Individual asset-state differences should be
judged against the unconditional and price-momentum baselines and the many comparisons
performed.

In [ ]:
point_statistics = regime_statistics(labels, point_regimes)
latest_statistics = regime_statistics(labels, latest_regimes)
relative_labels = return_spread_labels(
    labels,
    {
        "TIP minus IEF": ("TIP", "IEF"),
        "DBC minus IEF": ("DBC", "IEF"),
        "GLD minus IEF": ("GLD", "IEF"),
    },
)
primary_contrasts = regime_contrast_statistics(
    relative_labels,
    point_regimes,
    treated="accelerating",
    reference="decelerating",
)
display(point_statistics, primary_contrasts)
figure, _ = plot_regime_means(
    point_statistics,
    title="Inflation momentum and one-month outcomes",
)
plt.show()
plt.close(figure)

figure, _ = plot_regime_contrasts(
    primary_contrasts,
    title="Inflation-sensitive assets relative to nominal Treasuries",
)
plt.show()
plt.close(figure)

## Distributions and state balance

Inflation-sensitive assets can respond asymmetrically to shocks, real-yield changes,
and energy moves. Box plots therefore accompany means for `TIP`, `DBC`, `GLD`, and
nominal Treasuries. The count chart shows whether acceleration, stability, and
deceleration have adequate outcome coverage.

Extreme months are economically relevant and remain included. The plots are
descriptive and do not justify a post hoc winsorization rule.

Box plots retain all tail points but summarize quartiles rather than a full density.
Sample bars separately report outcome months and contiguous macro episodes after
masking unavailable labels. The headline/core agreement matrix is topic-specific: it
reveals whether changing the inflation measure changes state membership before any
outcome comparison is interpreted.

In [ ]:
figure, _ = plot_regime_distributions(
    labels[1],
    point_regimes,
    assets=("TIP", "DBC", "GLD", "IEF"),
    title="One-month outcomes by headline inflation momentum",
)
plt.show()
plt.close(figure)

figure, _ = plot_sample_sizes(point_statistics)
plt.show()
plt.close(figure)

## Sensitivity to the neutral band and inflation measure

The predeclared bands are 0.10, 0.25, and 0.50 percentage points. The primary heatmap
reports acceleration-minus-deceleration one-month contrasts for `TIP-IEF`, `DBC-IEF`,
and `GLD-IEF` under every headline and core band. This repeats exactly the primary
estimand instead of comparing an alternate raw summary table. It prevents a measure or
band from being selected solely because it produces a larger contrast.

The long table retains counts, outcome-eligible episodes, HAC errors, and intervals.
One exploratory Bonferroni family covers all measure-by-band-by-spread cells. The
heatmap is zero-centered and masks cells below the minimum-data display rule; the
underlying table still distinguishes missing estimation from an effect near zero.

In [ ]:
sensitivity_rows = []
for measure in ("CPIAUCSL", "CPILFESL"):
    for band in (0.10, 0.25, 0.50):
        regimes = momentum_regime(point_momentum[measure], band=band)
        table = regime_contrast_statistics(
            {1: relative_labels[1]},
            regimes,
            treated="accelerating",
            reference="decelerating",
        )
        table.insert(0, "band", band)
        table.insert(0, "measure", measure)
        sensitivity_rows.append(table)
sensitivity_table = simultaneous_interval_family(
    pd.concat(sensitivity_rows, ignore_index=True),
    family_id="inflation measure and threshold family",
)
sensitivity = sensitivity_table.pivot(
    index=["measure", "band"], columns="asset", values="mean_difference"
).where(
    sensitivity_table.pivot(
        index=["measure", "band"],
        columns="asset",
        values="meets_display_threshold",
    )
)
core_regimes = momentum_regime(point_momentum["CPILFESL"])
measure_agreement = classification_transition_table(
    point_regimes,
    core_regimes,
)
display(sensitivity_table, sensitivity, measure_agreement)
figure, _ = plot_sensitivity_heatmap(
    sensitivity,
    title="Headline and core acceleration-minus-deceleration contrasts",
    color_label="Difference in one-month mean return",
)
plt.show()
plt.close(figure)

figure, axis = plt.subplots(figsize=(7.5, 5.5))
agreement_image = axis.imshow(measure_agreement.to_numpy(), cmap="Blues")
axis.set(
    title="Headline and core momentum classification agreement",
    xlabel="Core classification",
    ylabel="Headline classification",
    xticks=range(len(measure_agreement.columns)),
    yticks=range(len(measure_agreement.index)),
    xticklabels=measure_agreement.columns,
    yticklabels=measure_agreement.index,
)
for row in range(len(measure_agreement.index)):
    for column in range(len(measure_agreement.columns)):
        axis.text(
            column,
            row,
            int(measure_agreement.iloc[row, column]),
            ha="center",
            va="center",
        )
figure.colorbar(agreement_image, ax=axis, label="Decision count")
figure.tight_layout()
plt.show()
plt.close(figure)

## Latest-revised bias diagnostic

Revised levels can change both year-over-year rates and six-month momentum. The
comparison reports regime membership and conditional-mean differences when today's
revisions are substituted into historical decisions. The plotted gap focuses on the
headline momentum feature and remains a retrospective diagnostic.

Material agreement or disagreement is retained without changing the primary feature
definition or threshold.

Latest-revised momentum uses the exact same four component months as each real-time
decision. The transition table includes unclassified states, so availability changes
are visible. The stability table is separate: it evaluates the real-time relative-return
contrast across halves and after leaving out each contiguous episode.

In [ ]:
revision_comparison = compare_statistics(point_statistics, latest_statistics)
classification_changes = classification_transition_table(
    point_regimes,
    latest_regimes,
)
stability = temporal_contrast_stability(
    relative_labels,
    point_regimes,
    treated="accelerating",
    reference="decelerating",
)
display(revision_comparison, classification_changes, stability)
figure, _ = plot_revision_gap(
    point_momentum["CPIAUCSL"],
    latest_momentum["CPIAUCSL"],
    title="Headline inflation-momentum revision substitution gap",
)
plt.show()
plt.close(figure)

## Limitations and live assertions

CPI is not an investor-specific consumption basket, and `TIP` returns reflect real
yields, duration, liquidity, and index mechanics as well as inflation accrual. Monthly
decisions omit release-time reactions. The fixed ETF universe has inception and
survivorship bias. Outcomes omit costs and taxes. Persistent regimes reduce effective
sample size, longer labels overlap, normal intervals are approximate, and the asset,
horizon, band, and inflation-measure family creates multiple-testing risk.

Execution assertions cover temporal availability, alignment, provenance, finite
summaries, and feature-label separation. They do not validate an economic story.

CPI is a revised index and current series-definition metadata do not prove historical
definition stability. Within-vintage arithmetic reduces that risk. The
[ALFRED real-time guide](https://fred.stlouisfed.org/docs/api/fred/realtime_period.html)
explains information vintages, while the
[Treasury TIPS page](https://www.treasurydirect.gov/marketable-securities/tips/)
explains principal indexation. Neither source implies that an ETF return must track
the inflation feature over a one-month market horizon.

In [ ]:
audit = validate_study_outputs(
    prices,
    point_in_time,
    labels,
    point_regimes,
    point_statistics,
    expected_regimes=frozenset({"accelerating", "stable", "decelerating"}),
    transformed=point_momentum_result,
)
assert_component_periods_match(point_rate_result, latest_rate_result)
assert_component_periods_match(point_momentum_result, latest_momentum_result)
assert set(point_in_time.frame.columns).isdisjoint(labels[1].frame.columns)
matched = point_in_time.provenance["available_from"].notna()
assert point_in_time.provenance.loc[matched, "available_from"].le(
    point_in_time.provenance.loc[matched, "decision_date"] - pd.Timedelta(days=1)
).all()
display(audit)
session.close()

## Interpretation after execution

Read feature coverage and regime counts before conditional means. Compare uncertainty
and distributions with both baselines, then inspect the complete band grid, the core
sensitivity, and latest-revised changes. Preserve null, unstable, and contrary
findings. These are historical associations, not proof that an asset is a reliable
inflation hedge or that the regimes support a profitable strategy.